<a href="https://colab.research.google.com/github/athinastavropoulou/ESG-AI-Analytics/blob/main/Marketing_Analytics_Paper_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai tomotopy nltk

import os
import glob
import wave
import tomotopy as tp
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from google import genai
from google.genai import types
import re

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

GEMINI_API_KEY = "YOUR_API_KEY_HERE"

sw = stopwords.words('english')
def tokenizer(doc, sw):
    return [word for word in word_tokenize(doc.lower()) if word not in sw and len(word) > 2]

def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
def process(folder_name, K):
    print(f"Ανάγνωση αρχείων από  '{folder_name}'")

    file_paths = glob.glob(f"{folder_name}/*.txt")

    if not file_paths:
        print(f"Δεν βρέθηκαν αρχεία .txt στον φάκελο '{folder_name}'!")
        return

    print(f"Βρέθηκαν {len(file_paths)} αρχεία.")

    lda = tp.LDAModel(k=K)

    for path in file_paths:
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                tokens = tokenizer(text, sw)
                if tokens:
                    lda.add_doc(tokens)
        except Exception as e:
            print(f"Σφάλμα στο αρχείο {path}: {e}")

    for i in range(0, 100, 10):
        lda.train(10)

    topics_summary = ""
    for k in range(lda.k):
        top_words = [pair[0] for pair in lda.get_topic_words(k, top_n=20)]
        topics_summary += f"Topic {k+1}: {', '.join(top_words)}\n"

    client = genai.Client(api_key=GEMINI_API_KEY)

    prompt_text = f"""
    I have performed LDA topic modeling on a set of documents.
    Here are the top 20 words for each of the {K} topics found:
    {topics_summary}

    Please:
    1. Write a single descriptive sentence for EACH topic based on its words.
    2. Combine these sentences into one flowing paragraph.
    3. Return ONLY the final paragraph inside <ANSWER> tags.
    """

    response_text = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt_text
    )

    result = response_text.text
    try:
        narration_text = re.search(r'<ANSWER>(.*?)</ANSWER>', result, re.DOTALL).group(1).strip()
    except:
        narration_text = result

    response_audio = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=narration_text,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Kore',
                    )
                )
            ),
        )
    )

    audio_data = response_audio.candidates[0].content.parts[0].inline_data.data
    output_filename = 'topics_narration.wav'
    wave_file(output_filename, audio_data)

    print(f"Ολοκληρώθηκε: '{output_filename}'")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompa

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
!unzip -o my_data.zip -d my_real_folder

process("docs", 5)